# Conteo de Colonias de Actinomicetos

## El problema

Contar colonias bacterianas en placas de Petri es una tarea de laboratorio muy comun. Sirve para saber cuantos microorganismos hay en una muestra. Hacerlo a mano tarda tiempo y los resultados pueden variar dependiendo de quien cuente. Por eso buscamos formas de automatizarlo.

En este proyecto trabajamos con imagenes de **actinomicetos**, un grupo de bacterias que forman colonias visibles en agar a simple vista. Cada imagen contiene **dos placas con la misma muestra**. Contar ambas y comparar los resultados sirve como indicador de confiabilidad: si los dos conteos son parecidos, el resultado es valido.

## Los datos

Tenemos 8 imagenes, cada una con dos placas (A y B). Los conteos manuales hechos en el laboratorio son los siguientes:

| Imagen | Placa A | Placa B |
|--------|---------|--------|
| actinomicetos_1 | 63 | 64 |
| actinomicetos_2 | 62 | 64 |
| actinomicetos_3 | 57 | 60 |
| actinomicetos_4 | 42 | 68 |
| actinomicetos_5 | 30 | 25 |
| actinomicetos_6 | 49 | 62 |
| actinomicetos_7 | 68 | 67 |
| actinomicetos_8 | 24 | 13 |

## Lo que se ha hecho hasta ahora

El proyecto parte de **CellSAM**, un modelo de segmentacion desarrollado por el Van Valen Lab (Caltech). El modelo combina dos partes:

- **AnchorDETR** detecta los objetos en la imagen y genera una caja alrededor de cada uno
- **SAM** (Segment Anything Model de Meta) toma cada caja y genera una mascara precisa pixel a pixel

CellSAM fue entrenado con mas de 8800 imagenes de microscopia. En este proyecto lo estamos adaptando para imagenes de placas de Petri, que son imagenes macroscopicas (visibles a simple vista), no microscopicas.

Para entender si CellSAM aporta algo en este contexto, lo comparamos con un metodo de **vision clasica**: un pipeline que no usa ninguna red neuronal, solo operaciones matematicas sobre los pixeles.

### El pipeline tiene tres etapas:

1. **Deteccion de placas:** encontrar donde estan los dos circulos en la imagen
2. **Segmentacion de colonias:** identificar cada colonia dentro de cada placa
3. **Filtrado:** descartar regiones que no son colonias (muy pequenas, muy grandes, o con forma irregular)

## 1. Importaciones

In [ ]:
import sys
import csv
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops, label as sk_label
from cellSAM import get_model, segment_cellular_image

IMAGES_DIR = Path('images/placas')
GT_CSV     = IMAGES_DIR / 'ground_truth.csv'
OUTPUT_DIR = Path('results/colonies')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Librerias cargadas.')

## 2. Parametros

In [ ]:
MIN_COLONY_AREA    = 300
MAX_COLONY_AREA    = 50000
MIN_SOLIDITY       = 0.50

TOPHAT_KERNEL      = 81
TOPHAT_THRESHOLD   = 10
SAT_THRESHOLD      = 60
MAX_HOLE_AREA      = 400
OPEN_ITERATIONS    = 1
WATERSHED_MIN_DIST = 20

NORMALIZE          = True
POSTPROCESS        = False

print('Parametros definidos.')

## 3. Cargar el modelo CellSAM

`get_model()` descarga los pesos si no estan en cache o los carga desde `~/.deepcell/models/`. Se carga una sola vez y se reutiliza para todas las imagenes.

In [ ]:
print('Cargando modelo CellSAM...')
model = get_model()
print('Modelo listo.')

## 4. Deteccion de placas

Usamos **HoughCircles** para encontrar los dos circulos en cada imagen. Dividimos la imagen en dos mitades antes de buscar para evitar que detecte dos circulos en la misma placa. Luego recortamos cada placa y ponemos a negro todo lo que esta fuera del circulo.

In [ ]:
def detect_plates(img_bgr):
    h, w     = img_bgr.shape[:2]
    portrait = h > w
    plates   = []
    for i in range(2):
        if portrait:
            y0, y1 = i * h // 2, (i + 1) * h // 2
            half   = img_bgr[y0:y1, :]
            ox, oy = 0, y0
        else:
            x0, x1 = i * w // 2, (i + 1) * w // 2
            half   = img_bgr[:, x0:x1]
            ox, oy = x0, 0
        hh, hw  = half.shape[:2]
        gray    = cv2.cvtColor(half, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (21, 21), 0)
        circles = cv2.HoughCircles(
            blurred, cv2.HOUGH_GRADIENT, dp=1.2,
            minDist=max(hh, hw), param1=60, param2=25,
            minRadius=int(min(hh, hw) * 0.30),
            maxRadius=int(min(hh, hw) * 0.52),
        )
        if circles is not None:
            best = np.round(circles[0][0]).astype(int)
            cx, cy, r = int(best[0]) + ox, int(best[1]) + oy, int(best[2])
        else:
            cx, cy, r = hw // 2 + ox, hh // 2 + oy, int(min(hh, hw) * 0.43)
        plates.append((cx, cy, r))
    return plates

def crop_plate(img_bgr, cx, cy, r, shrink=0.86):
    r_use = int(r * shrink)
    x1 = max(0, cx - r_use);  y1 = max(0, cy - r_use)
    x2 = min(img_bgr.shape[1], cx + r_use)
    y2 = min(img_bgr.shape[0], cy + r_use)
    crop   = img_bgr[y1:y2, x1:x2].copy()
    hc, wc = crop.shape[:2]
    mask   = np.zeros((hc, wc), dtype=np.uint8)
    cv2.circle(mask, (cx - x1, cy - y1), r_use, 255, -1)
    crop[mask == 0] = 0
    return crop, mask

print('Funciones de deteccion definidas.')

## 5. Metodo clasico: vision sin redes neuronales

Detecta colonias usando operaciones matematicas sobre los pixeles, paso a paso:

1. **Gaussian blur 5x5** para suavizar el ruido de la camara
2. **Top-hat morfologico en escala de grises (kernel 81px):** extrae manchas que son mas brillantes que su entorno local, sin importar la iluminacion general de la placa
3. **Umbral fijo sobre el top-hat:** convierte el resultado en blanco y negro. Todo lo que supera el umbral es posible colonia
4. **Umbral de saturacion HSV:** detecta colonias de color (rosa, naranja, amarillo) que no son mas brillantes que el fondo pero si mas saturadas
5. **OR de ambas mascaras:** una region es colonia si es brillante O si tiene color
6. **Relleno de huecos pequenos (menos de 400 px2):** muchas colonias tienen un punto oscuro en el centro. Sin esto aparecen como anillos en lugar de discos
7. **Apertura morfologica:** elimina puntos de ruido
8. **Watershed:** separa colonias que se tocan usando la distancia al borde para encontrar los centros

In [ ]:
def segment_classical(crop_bgr, plate_mask):
    gray    = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    gray    = cv2.bitwise_and(gray, gray, mask=plate_mask)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (TOPHAT_KERNEL, TOPHAT_KERNEL))
    tophat  = cv2.morphologyEx(blurred, cv2.MORPH_TOPHAT, kernel)
    _, bin_lum = cv2.threshold(tophat, TOPHAT_THRESHOLD, 255, cv2.THRESH_BINARY)
    bin_lum    = cv2.bitwise_and(bin_lum, bin_lum, mask=plate_mask)
    hsv        = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    sat        = cv2.bitwise_and(hsv[:, :, 1], hsv[:, :, 1], mask=plate_mask)
    _, bin_col = cv2.threshold(sat, SAT_THRESHOLD, 255, cv2.THRESH_BINARY)
    binary     = cv2.bitwise_or(bin_lum, bin_col)
    binary     = cv2.bitwise_and(binary, binary, mask=plate_mask)
    inv_labeled   = sk_label(~binary.astype(bool))
    binary_filled = binary.copy()
    for rid in range(1, inv_labeled.max() + 1):
        hole = inv_labeled == rid
        if hole.sum() <= MAX_HOLE_AREA:
            binary_filled[hole] = 255
    k3            = np.ones((3, 3), np.uint8)
    binary_filled = cv2.morphologyEx(binary_filled, cv2.MORPH_OPEN, k3, iterations=OPEN_ITERATIONS)
    binary_filled = cv2.bitwise_and(binary_filled, binary_filled, mask=plate_mask)
    dist   = ndimage.distance_transform_edt(binary_filled)
    coords = peak_local_max(dist, min_distance=WATERSHED_MIN_DIST,
                            threshold_rel=0.15, labels=binary_filled.astype(bool))
    if len(coords) == 0:
        _, labels_out = cv2.connectedComponents(binary_filled)
        props = regionprops(labels_out)
        valid = [p for p in props if MIN_COLONY_AREA <= p.area <= MAX_COLONY_AREA
                 and p.solidity >= MIN_SOLIDITY]
        return labels_out, valid
    local_max              = np.zeros_like(dist, dtype=bool)
    local_max[tuple(coords.T)] = True
    markers   = sk_label(local_max)
    labels_ws = watershed(-dist, markers, mask=binary_filled.astype(bool))
    props = regionprops(labels_ws)
    valid = [p for p in props if MIN_COLONY_AREA <= p.area <= MAX_COLONY_AREA
             and p.solidity >= MIN_SOLIDITY]
    return labels_ws, valid

print('Funcion clasica definida.')

## 6. Metodo CellSAM

En lugar de operar pixel a pixel, CellSAM entiende la imagen como un todo. Recibe el recorte de la placa, genera cajas alrededor de cada objeto y produce una mascara por objeto. El parametro `normalize=True` aplica normalizacion por percentil y CLAHE antes de pasar la imagen al modelo.

In [ ]:
def segment_cellsam(crop_bgr, plate_mask):
    crop_rgb   = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    mask, _, _ = segment_cellular_image(
        crop_rgb, model=model,
        normalize=NORMALIZE, postprocess=POSTPROCESS, device='cpu',
    )
    mask = mask.copy()
    mask[plate_mask == 0] = 0
    props = regionprops(mask)
    valid = [p for p in props if MIN_COLONY_AREA <= p.area <= MAX_COLONY_AREA
             and p.solidity >= MIN_SOLIDITY]
    return mask, valid

def draw_overlay(crop_bgr, valid_props, label_mask):
    overlay = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    for prop in valid_props:
        region = label_mask == prop.label
        overlay[region] = overlay[region] * 0.45 + np.array([0.15, 0.85, 0.35]) * 0.55
    return overlay

print('Funciones CellSAM definidas.')

## 7. Cargar los conteos manuales (ground truth)

In [ ]:
gt = {}
with open(GT_CSV, newline='', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        stem = Path(row['image']).stem
        gt[stem] = (int(row['plate_A']), int(row['plate_B']))

print(f'Ground truth cargado: {len(gt)} imagenes')
for k, v in gt.items():
    print(f'  {k}: A={v[0]}, B={v[1]}')

## 8. Resultados por imagen

Para cada imagen se muestran las dos placas con las colonias detectadas por cada metodo. Las regiones en verde son las colonias que cada metodo encontro y paso el filtro de tamano y forma.

In [ ]:
SUPPORTED = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
images    = sorted(p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in SUPPORTED)
print(f'Imagenes encontradas: {len(images)}\n')

results = []

for idx, img_path in enumerate(images, 1):
    img    = cv2.imread(str(img_path))
    plates = detect_plates(img)

    counts_cl, counts_cs = [], []

    # Una figura por imagen: 2 filas (placa A y B) x 3 columnas (original | clasico | CellSAM)
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    gt_a, gt_b = gt.get(img_path.stem, (None, None))
    gt_str = f'  (GT: A={gt_a}, B={gt_b})' if gt_a is not None else ''
    fig.suptitle(f'{img_path.name}{gt_str}', fontsize=13, fontweight='bold')

    for i, (cx, cy, r) in enumerate(plates):
        name       = ['A', 'B'][i]
        gt_val     = (gt_a, gt_b)[i]
        crop, mask = crop_plate(img, cx, cy, r)

        lbl_cl, val_cl = segment_classical(crop, mask)
        lbl_cs, val_cs = segment_cellsam(crop, mask)

        n_cl, n_cs = len(val_cl), len(val_cs)
        counts_cl.append(n_cl)
        counts_cs.append(n_cs)

        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

        gt_label = f' (GT={gt_val})' if gt_val is not None else ''
        axes[i, 0].imshow(crop_rgb)
        axes[i, 0].set_title(f'Placa {name} original{gt_label}', fontsize=11)
        axes[i, 0].axis('off')

        err_cl = f'  err={n_cl - gt_val:+d}' if gt_val is not None else ''
        axes[i, 1].imshow(draw_overlay(crop, val_cl, lbl_cl))
        axes[i, 1].set_title(f'Clasico: {n_cl} colonias{err_cl}', fontsize=11)
        axes[i, 1].axis('off')

        err_cs = f'  err={n_cs - gt_val:+d}' if gt_val is not None else ''
        axes[i, 2].imshow(draw_overlay(crop, val_cs, lbl_cs))
        axes[i, 2].set_title(f'CellSAM: {n_cs} colonias{err_cs}', fontsize=11)
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{img_path.stem}_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

    results.append({
        'name':      img_path.stem,
        'classical': counts_cl,
        'cellsam':   counts_cs,
        'gt':        gt.get(img_path.stem),
    })

print('Procesamiento completado.')

## 9. Tabla de resultados

In [ ]:
rows = []
for r in results:
    ga, gb = r['gt'] if r['gt'] else (None, None)
    ca, cb = r['classical']
    sa, sb = r['cellsam']
    rows.append({
        'Imagen':          r['name'],
        'GT A':            ga,
        'GT B':            gb,
        'Clasico A':       ca,
        'Clasico B':       cb,
        'CellSAM A':       sa,
        'CellSAM B':       sb,
        'Err Clasico A':   (ca - ga) if ga is not None else None,
        'Err Clasico B':   (cb - gb) if gb is not None else None,
        'Err CellSAM A':   (sa - ga) if ga is not None else None,
        'Err CellSAM B':   (sb - gb) if gb is not None else None,
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / 'comparison_summary.csv', index=False)
df

## 10. Grafico comparativo

Tres barras por imagen: verde = conteo manual, azul = metodo clasico, naranja = CellSAM.

In [ ]:
names = [r['name'].replace('actinomicetos_', 'actin_') for r in results]
x     = np.arange(len(results))
width = 0.22

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 10), sharex=True)
fig.suptitle('Comparativa de metodos de conteo de colonias', fontsize=14, fontweight='bold')

for ax, plate_idx, plate_label in [(ax1, 0, 'Placa A'), (ax2, 1, 'Placa B')]:
    gt_vals = [r['gt'][plate_idx] if r['gt'] else 0 for r in results]
    cl_vals = [r['classical'][plate_idx]             for r in results]
    cs_vals = [r['cellsam'][plate_idx]               for r in results]

    b0 = ax.bar(x - width,  gt_vals, width, label='Ground truth', color='#2ca02c', alpha=0.85)
    b1 = ax.bar(x,          cl_vals, width, label='Clasico CV',   color='#4C72B0', alpha=0.85)
    b2 = ax.bar(x + width,  cs_vals, width, label='CellSAM',      color='#DD8452', alpha=0.85)
    for bars in [b0, b1, b2]:
        ax.bar_label(bars, padding=3, fontsize=8)
    ax.set_ylabel('Colonias contadas')
    ax.set_title(plate_label, fontsize=12)
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(gt_vals + cl_vals + cs_vals, default=1) * 1.25)
    ax.grid(axis='y', alpha=0.3)

ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Grafico guardado en {OUTPUT_DIR}/comparison_chart.png')

## 11. Error promedio por metodo

In [ ]:
errs_cl, errs_cs = [], []
for r in results:
    if not r['gt']:
        continue
    for i in range(2):
        errs_cl.append(abs(r['classical'][i] - r['gt'][i]))
        errs_cs.append(abs(r['cellsam'][i]   - r['gt'][i]))

fig, ax = plt.subplots(figsize=(6, 4))
metodos = ['Clasico CV', 'CellSAM']
errores = [np.mean(errs_cl), np.mean(errs_cs)]
colores = ['#4C72B0', '#DD8452']
bars = ax.bar(metodos, errores, color=colores, alpha=0.85, width=0.4)
ax.bar_label(bars, fmt='%.1f', padding=4, fontsize=11)
ax.set_ylabel('Error absoluto medio (colonias)')
ax.set_title('Error promedio vs ground truth por metodo')
ax.set_ylim(0, max(errores) * 1.4)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('=' * 45)
print('  ERROR ABSOLUTO MEDIO vs GROUND TRUTH')
print('=' * 45)
print(f'  Metodo clasico: {np.mean(errs_cl):.1f} colonias por placa')
print(f'  CellSAM:        {np.mean(errs_cs):.1f} colonias por placa')
print('=' * 45)

## 12. Que encontramos

Ambos metodos intentan detectar colonias en imagenes para las que no fueron disenados originalmente:

- El **metodo clasico** fue ajustado manualmente con parametros que funcionan para colonias blancas o crema. Es rapido y no requiere GPU, pero depende de que las condiciones de imagen sean parecidas entre fotos.

- **CellSAM** fue entrenado con imagenes de microscopia, no con placas de Petri macroscopicas. Aun asi puede detectar objetos circulares con bordes definidos, que es lo que son las colonias. Su ventaja es que no necesita parametros ajustados a mano.

La comparativa con el ground truth nos muestra que tan lejos esta cada metodo del conteo real.

## 13. Pasos a futuro

### Opcion 1: Ajustar parametros del metodo clasico
Si el metodo clasico queda cerca del ground truth, se pueden ajustar `TOPHAT_THRESHOLD`, `WATERSHED_MIN_DIST` y `MIN_COLONY_AREA` para mejorar los casos donde falla. Es la opcion mas rapida.

### Opcion 2: Ajustar parametros de CellSAM
CellSAM tiene parametros sin explorar:
- `postprocess=True`: postprocesamiento extra para imagenes ruidosas
- `bbox_threshold`: que tan seguros deben ser los bounding boxes (default 0.4)
- `normalize=False`: desactiva el preprocesamiento interno

### Opcion 3: Fine-tuning de CellSAM
Si ninguno de los dos metodos llega al nivel deseado, se puede hacer fine-tuning de CellSAM con imagenes de actinomicetos etiquetadas. Esto requiere crear mascaras de segmentacion para cada colonia y entrenar el modelo unas pocas iteraciones. Con pocas imagenes bien anotadas puede mejorar significativamente el rendimiento en este tipo de placa.

### Opcion 4: Expandir a otras bacterias
Una vez que el pipeline funciona bien para actinomicetos, se puede adaptar para otras bacterias ajustando los parametros de tamano y forma de las colonias.